In [1]:
import {
  REGISTRAR_USERNAME,
  REGISTRAR_PASSWORD
} from '../commonHelpers/vars.ts'
import { authenticate } from '../commonHelpers/authentication.ts'

const token = await authenticate(REGISTRAR_USERNAME, REGISTRAR_PASSWORD)
token

const payload = JSON.parse(atob(token.split('.')[1]))
const userId = payload.sub
userId


"6509920c42f19d1950648ea8"

In [ ]:
import { registerSystem } from '../commonHelpers/gqlHandlers.ts'
import { getTokenForSystemClient } from '../commonHelpers/authentication.ts'
import {
  CLIENT_ID,
  CLIENT_SECRET,
  ADMIN_USERNAME,
  ADMIN_PASSWORD
} from '../commonHelpers/vars.ts'

let clientId: string | null = null
let clientSecret: string | null = null

if (CLIENT_ID && CLIENT_SECRET) {
  clientId = CLIENT_ID
  clientSecret = CLIENT_SECRET
} else {
  // For dev mode
  const adminToken = await authenticate(ADMIN_USERNAME, ADMIN_PASSWORD)
  const systemRegistration = await registerSystem(adminToken)

  clientId = systemRegistration.data.registerSystem.system.clientId
  clientSecret = systemRegistration.data.registerSystem.clientSecret
}

const sysToken = await getTokenForSystemClient(clientId, clientSecret)
sysToken


In [2]:
import { getLocations } from '../commonHelpers/locations.ts'

const locations = await getLocations(token, 'ADMIN_STRUCTURE')
const healthFacilities = await getLocations(token, 'HEALTH_FACILITY')
const crvsOffices = await getLocations(token, 'CRVS_OFFICE')
const locationsData = await locations.json()
const locationIds = locationsData.entry.map((loc: any) => loc.resource.id)
const healthFacilitiesData = await healthFacilities.json()
const healthFacilityIds = healthFacilitiesData.entry.map((loc: any) => loc.resource.id)
const crvsOfficesData = await crvsOffices.json()
const crvsOfficeIds = crvsOfficesData.entry.map((loc: any) => loc.resource.id)


In [ ]:
import { bulkImport, getIndexErrors } from './helpers/gqlHelpers.ts'
import { batch } from '../commonHelpers/utils.ts'

const BATCH_SIZE = 100

const batchImport = async (batch: any[], sysToken: string) => {
  const res = await bulkImport(batch, sysToken)
  const errors = getIndexErrors(res)
  if (errors) {
    console.error('Errors during bulk import', errors)
  }
}

const migrateEvents = async (events: any[], type: string) => {
  let imported = 0
  let remaining = events.length

  const batches = batch(events, BATCH_SIZE)

  console.log('Found ', events.length, ` ${type} events to import`)

  for (const batch of batches) {
    console.log(`Importing next ${batch.length} of remaining ${remaining}...`)
    await batchImport(batch, sysToken)

    imported += batch.length
    remaining -= batch.length
  }
  console.log(`Imported ${imported} of ${events.length}...`)
}


In [ ]:
import { createBulkEvents } from './generators/bulkEventGenerator.ts'

const NUMBER_TO_GENERATE = 2
const START_FROM = 0 // To avoid tracking number collisions

const events = createBulkEvents(
  locationIds,
  healthFacilityIds,
  crvsOfficeIds,
  NUMBER_TO_GENERATE,
  START_FROM,
  userId,
  'REGISTRAR'
)

await migrateEvents(events, 'CRVS')



[
  {
    id: "22262c9a-b1aa-46f6-b734-e9e72d3c3c92",
    type: "birth",
    createdAt: "2024-09-17",
    updatedAt: "2024-09-17",
    updatedAtLocation: "140ca057-98ed-42b1-899c-1a752be12281",
    trackingId: "BI0000000",
    actions: [
      {
        type: "CREATE",
        createdAt: "2024-09-17",
        createdBy: "6509920c42f19d1950648ea8",
        createdByUserType: "user",
        createdByRole: "REGISTRAR",
        createdAtLocation: "140ca057-98ed-42b1-899c-1a752be12281",
        updatedAtLocation: "140ca057-98ed-42b1-899c-1a752be12281",
        status: "Accepted",
        declaration: {},
        id: "2d03b84c-304e-41fd-b9ad-1a3c30ac910e",
        transactionId: "9b683566-22c0-4439-a1eb-41536c34881d"
      },
      {
        id: "c5c43050-0e82-4c7c-bdf0-1fa2af786cbe",
        type: "REGISTER",
        transactionId: "da8d7f54-1dde-4583-8c4f-befba69058b7",
        createdAt: "2024-09-17",
        createdBy: "6509920c42f19d1950648ea8",
        createdByUserType: "user",
     